<center>
<h2> </h2>
<h2>ALTeGraD 2024<br>
<h2>Lab Session 3: Large Language Models</h2>
<h5>October 22, 2024</h5>
<h4><b>Student Name: Antoine Peyronnet</b> </h4>
<br>
</center>
[Qwen2.5-0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B)
<hr style="border:10px solid gray"> </hr>
<p style="text-align: justify;">
This handout includes theoretical introductions, <font color='blue'>coding tasks</font> and <font color='red'>questions</font>. Before the deadline, you should submit a <B>.ipynb</B> file named <b>Lastname_Firstname.ipynb</b> containing your notebook (with the gaps filled and your answers to the questions). Your answers should be well constructed and well justified. They should not repeat the question or generalities in the handout. When relevant, you are welcome to include figures, equations and tables derived from your own computations, theoretical proofs or qualitative explanations. One submission is required for each student. The deadline for this lab is <b>October 29, 2024 11:59 PM</b>. No extension will be granted. Late policy is as follows: ]0, 24] hours late → -5 pts; ]24, 48] hours late → -10 pts; > 48 hours late → not graded (zero).

<hr style="border:5px solid gray"> </hr>

<hr style="border:5px solid gray"> </hr>

* Please submit your file to Moodle or [here](https://docs.google.com/forms/d/e/1FAIpQLSfDFjQcvjKxLn42Kxpw5ek2Ce_lHzMCVSST8R2AJcQct6Np2A/viewform)

<br><br>
In this lab, we will:

* fintune [Qwen2.5-0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B) on a question/answer dataset.

* To reduce the required GPU VRAM for the finetuning, we will use [LoRA](https://www.anyscale.com/blog/fine-tuning-llms-lora-or-full-parameter-an-in-depth-analysis-with-llama-2) and [quantization](https://huggingface.co/blog/4bit-transformers-bitsandbytes) techniques.

* Compare the results before and after instructin tuning.

* Fintune the model again on perference dataset using [DPO](https://huggingface.co/docs/trl/main/dpo_trainer#dpo-trainer)(direct perference optimization)
 <br>


# <b>Part 1 Finetuning Qwen2.5-0.5B using HuggingFace's Transfromers</b>
In this section, we will fintune [Qwen2.5-0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B) on a question/answer dataset.

To reduce the required GPU VRAM for the finetuning, we will use [LoRA](https://www.anyscale.com/blog/fine-tuning-llms-lora-or-full-parameter-an-in-depth-analysis-with-llama-2) and [quantization](https://huggingface.co/blog/4bit-transformers-bitsandbytes) techniques.

## <b>Preparing the environment and installing libraries:<b>

In [1]:
!nvidia-smi

Tue Oct 29 20:55:48 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   45C    P8               9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
!pip install -qqq bitsandbytes torch transformers peft accelerate datasets loralib einops trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 9.2 MB/s eta 0:00:00


In [3]:
!pip install --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 19.2 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.44.2
    Uninstalling transformers-4.44.2:
      Successfully uninstalled transformers-4.44.2


In [ ]:
!pip install transformers==4.45.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 34.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.46.1
    Uninstalling transformers-4.46.1:
      Successfully uninstalled transformers-4.46.1


After having dealt with a serious of misfortunes in the dpo training I understood that I had to upgrade my transformer version to something else:

In [4]:
!pip install git+https://github.com/huggingface/trl


  Cloning https://github.com/huggingface/trl to /tmp/pip-req-build-hottthgp
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/trl /tmp/pip-req-build-hottthgp
  Resolved https://github.com/huggingface/trl to commit b2696578ce6db1749a250661b507bf8b90e14dd5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for trl: filename=trl-0.12.0.dev0-py3-none-any.whl size=309902 sha256=8be9510cdf16839c42c66b7f925ef8b0e97db44945074f329d57b3f8cfb629f7
  Stored in directory: /tmp/pip-ephem-wheel-cache-xpzusx5r/wheels/6a/aa/56/d64d9ae3521350622f9325fdc3bccb4dd3d3ec1c1d8e917400
Successfully built trl
  Attempting uninstall: trl
    Found existing installation: trl 0.11.4
    Uninstalling trl-0.11.4:
      Successfully uninstalled trl-0.11.4


In [5]:
import json
import os
from pprint import pprint

import bitsandbytes as bnb
import pandas as pd
import torch
import torch.nn as nn
import transformers
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer

from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

## <b>Loading the model and the tokenizer:<b>

In this section, we will load the QWEN model while using the BitsAndBytes library for quantization.

In [6]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
# MODEL_NAME = "unsloth/Llama-3.2-1B" # Try Llama if you want

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)# fill the gap

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=bnb_config,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Here: we are considering

In [7]:
def print_trainable_parameters(model):

    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()# fill the gap: get the number of trainable parameters: trainable_params
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

## <b>Configuring LoRA:<b>

Since we would like to work on tasks related to text generation we choose: `TaskType.CAUSAL_LM`

In [8]:
from peft import LoraConfig, TaskType, get_peft_model

In [9]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # target_modules=["query_key_value"],
    bias="none",
    task_type= TaskType.CAUSAL_LM# fill the gaph
)

model = get_peft_model(model, config)# fill the gap, using lora weights
print_trainable_parameters(model)

trainable params: 1081344 || all params: 495114112 || trainable%: 0.21840298504761665


## <b>Test the model before finetuning:<b>

In [10]:
prompt = "<human>: What equipment do I need for rock climbing?  \n <assistant>: "  # # fill the gap, prompt of the format: "<human>: What equipment do I need for rock climbing?  \n <assistant>: ", with an empty response from the assistant
print(prompt)


generation_config = model.generation_config
generation_config.max_new_tokens = 200
generation_config.temperature = 0.7
generation_config.top_p = 0.7
generation_config.num_return_sequences = 1
generation_config.pad_token_id = tokenizer.eos_token_id
generation_config.eos_token_id = tokenizer.eos_token_id

<human>: What equipment do I need for rock climbing?  
 <assistant>: 


Question: what does the temperature do in the above cell?

The temperature is a parameter used in order to adjusts probability of what comes next.

Let's have this image with the way we as humans answer a question for instance:
we could think that when we are answering, we could think about other words, other stories to tell; and that would be ground to extrapolate and give other examples. So in a way we could add randomness in our answer by building bonds with other ideas we have when answering.
That illustration proposes some ideas of what temperature is. It's a feature that could adjust balance between coherence and randomness.

Setting temperature to 1 (default) is like keeping a balance between randomness and coherence.
Setting it less than 1 like what we are doing (0.7) gives us a more deterministic answer, more facts.
Setting it higher than 1 allows more possibilities to have answers including irrelevant content

In [11]:
%%time
device = "cuda:0"

encoding = tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = model.generate(
        input_ids=encoding.input_ids,
        attention_mask=encoding.attention_mask,
        generation_config=generation_config,
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


<human>: What equipment do I need for rock climbing?  
 <assistant>: 1. Climbing harness, 2. Climbing rope, 3. Climbing shoes, 4. Climbing gloves, 5. Climbing helmet, 6. Climbing harness, 7. Climbing harness, 8. Climbing harness, 9. Climbing harness, 10. Climbing harness, 11. Climbing harness, 12. Climbing harness, 13. Climbing harness, 14. Climbing harness, 15. Climbing harness, 16. Climbing harness, 17. Climbing harness, 18. Climbing harness, 19. Climbing harness, 20. Climbing harness, 21. Climbing harness, 22. Climbing harness, 23. Climbing harness, 24. Climbing harness, 25. Climbing harness, 26. Climbing harness, 2
CPU times: user 47.5 s, sys: 170 ms, total: 47.7 s
Wall time: 53.2 s


## <b>Loading the question/answer dataset from HuggingFace:<b>

In [12]:
data = load_dataset("HuggingFaceH4/helpful-instructions")
pd.DataFrame(data["train"])

README.md:   0%|          | 0.00/2.05k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/20.8M [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/12.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/147706 [00:00<?, ? examples/s]

,instruction,demonstration,meta
0,"Hi, I want to learn to play horseshoes. Can yo...","I can, but maybe I should begin by telling you...",{'source': 'helpful-anthropic-raw'}
1,How do I teach kids to meditate?,Great question! That’s a really useful skill t...,{'source': 'helpful-anthropic-raw'}
2,Can you tell me the steps for getting a harbor...,Sure. I believe you’ll need a copy of the mari...,{'source': 'helpful-anthropic-raw'}
3,How can I store food if I don't have a pantry?,"You could store the food in a refrigerator, th...",{'source': 'helpful-anthropic-raw'}
4,what are some good novels for a 9 year old?,"That depends on the 9 year old, but if they li...",{'source': 'helpful-anthropic-raw'}
...,...,...,...
147701,"Given the following sentence, classify it into...",Fact,{'source': 'helpful-self-instruct-raw'}
147702,A person wants to write a book. he/she writes ...,Chapter 1 - The History of China\nChapter 2 - ...,{'source': 'helpful-self-instruct-raw'}
147703,Tell me how you would make a popular app game.,I would make a game that is similar to 2048. T...,{'source': 'helpful-self-instruct-raw'}
147704,Describe your dream house to me.\n\nOutput:,My dream house is a two-story building with a ...,{'source': 'helpful-self-instruct-raw'}


## <b>Preparing the finetuning data:<b>

In [13]:
print(data["train"][0])


{'instruction': 'Hi, I want to learn to play horseshoes. Can you teach me?', 'demonstration': 'I can, but maybe I should begin by telling you that a typical game consists of 2 players and 6 or 8 horseshoes.', 'meta': {'source': 'helpful-anthropic-raw'}}


In [14]:
def generate_prompt(data_point):
    question = data_point["instruction"]
    answer = data_point["demonstration"]
    prompt = f"<human>: {question}  \n<assistant>: {answer}"
    return prompt # fill the gap, transform the data into prompts of the format: "<human>: question?  \n <assistant>: response"

def generate_and_tokenize_prompt(data_point):
    full_prompt = generate_prompt(data_point)
    tokenized_full_prompt = tokenizer(full_prompt, padding=True, truncation=True)
    return tokenized_full_prompt

data = data["train"].shuffle(seed=42).map(generate_and_tokenize_prompt)

Map:   0%|          | 0/147706 [00:00<?, ? examples/s]

## <b>Finetuning:<b>

In [15]:
OUTPUT_DIR = "experiments"

training_args = transformers.TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    save_total_limit=3,
    logging_steps=1,
    output_dir=OUTPUT_DIR,
    max_steps=200,   # try more steps if you can
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="tensorboard",
)

trainer = transformers.Trainer(
    model=model,
    train_dataset=data,
    args=training_args,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

model.config.use_cache = False
trainer.train()

/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss
1,3.415900
2,1.621000
3,2.598100
4,1.884800
5,2.734500
6,2.563300
7,3.078600
8,2.185600
9,2.960700
10,2.515200


TrainOutput(global_step=200, training_loss=1.9429401087760925, metrics={'train_runtime': 345.1124, 'train_samples_per_second': 2.318, 'train_steps_per_second': 0.58, 'total_flos': 156500683637760.0, 'train_loss': 1.9429401087760925, 'epoch': 0.005416164543078819})

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir experiments/runs --port 6008

## <b>Test the model after the finetuning:<b>

In [16]:
%%time
device = "cuda:0"

encoding = tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = model.generate(
        input_ids=encoding.input_ids,
        attention_mask=encoding.attention_mask,
        generation_config=generation_config,
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


<human>: What equipment do I need for rock climbing?  
 <assistant>:  Rock climbing requires a lot of equipment, including a harness, a rope, a climbing harness, a climbing rope, a climbing ladder, a climbing wall, a climbing wall ladder, a climbing wall ladder stand, a climbing wall ladder stand stand, a climbing wall ladder stand stand stand, a climbing wall ladder stand stand stand stand, a climbing wall ladder stand stand stand stand stand, a climbing wall ladder stand stand stand stand stand stand, a climbing wall ladder stand stand stand stand stand stand stand, a climbing wall ladder stand stand stand stand stand stand stand stand, a climbing wall ladder stand stand stand stand stand stand stand stand stand, a climbing wall ladder stand stand stand stand stand stand stand stand stand, a climbing wall ladder stand stand stand stand stand stand stand stand stand stand, a climbing wall ladder stand stand stand stand stand stand stand stand stand stand stand, a climbing wall ladder 

In [17]:
def generate_response(question: str) -> str:
    prompt = f" <human>: {question}? \n <assistant>: "# fill the gap, transform the data into prompts of the format: "<human>: question?  \n <assistant>: " with an empty response
    encoding = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=encoding.input_ids,
            attention_mask=encoding.attention_mask,
            generation_config=generation_config,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    assistant_start = "<assistant>:"
    response_start = response.find(assistant_start)
    return response[response_start + len(assistant_start) :].strip()

In [18]:
prompt = "What program can I use to edit video clips I took with my phone?"
print('-', prompt,'\n')
print(generate_response(prompt))

prompt = "Do you know the reasons as to why people love coffee so much?"
print('\n\n\n-', prompt, '\n')
print(generate_response(prompt))



- What program can I use to edit video clips I took with my phone? 

You can use the built-in video editing software on your phone. You can also use third-party apps like Adobe Premiere Pro or Final Cut Pro. You can also use third-party apps like iMovie or iMovie Pro to edit video clips. You can also use third-party apps like iMovie or iMovie Pro to edit video clips. You can also use third-party apps like iMovie or iMovie Pro to edit video clips. You can also use third-party apps like iMovie or iMovie Pro to edit video clips. You can also use third-party apps like iMovie or iMovie Pro to edit video clips. You can also use third-party apps like iMovie or iMovie Pro to edit video clips. You can also use third-party apps like iMovie or iMovie Pro to edit video clips. You can also use third-party apps like iMovie or iMovie Pro to edit video clips. You can also use third-party apps like iMovie or iMovie Pro to edit video clips.



- Do you know the reasons as to why people love coffee so mu

Remark: It's strange I had another result without sentence that repeat themselves.

But anyways, here we can see that the model generates responses that repeat themselves. But the sentences are still understandable.

We could refine our training and probably add some penalty in order to deal with those problems

# Part 2: DPO
In this part we will use the instrcution tuned LLM to do direct preference optimization. see the paper: https://arxiv.org/abs/2305.18290

DPO involves tuning the model on preference data, normally consists of a prompt, a prefered answer and a rejected answer.

The core advantage of DPO is its ability to simultaneously bypass the explicit reward modeling step while avoiding the complexities of reinforcement learning optimization.

## Test the model before DPO:


In [19]:
prompt_2 = "<system> You are a helpful assistant \n<human>: Can you taste this dish and tell me if it needs more spices?  \n<assistant>: "
print(prompt_2)

<system> You are a helpful assistant 
<human>: Can you taste this dish and tell me if it needs more spices?  
<assistant>: 


In [20]:
%%time
device = "cuda:0"

encoding = tokenizer(prompt_2, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = model.generate(
        input_ids=encoding.input_ids,
        attention_mask=encoding.attention_mask,
        generation_config=generation_config,
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<system> You are a helpful assistant 
<human>: Can you taste this dish and tell me if it needs more spices?  
<assistant>:  I can’t taste it, but I can tell you that it’s a good dish.  It’s a good dish, and it’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.  It’s a good dish.
CPU times: user 45.1 s, sys: 88.8 ms, total: 45.2 s
Wall time: 45.5 s


## Loading the preference data from Huggingface:

In [21]:
data_dpo = load_dataset("CultriX/llama70B-dpo-dataset")
pd.DataFrame(data_dpo["train"])

README.md:   0%|          | 0.00/171 [00:00<?, ?B/s]

dataset_final_v2.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2179 [00:00<?, ? examples/s]

,system,question,chosen,rejected
0,You are a helpful assistant,Can you taste this dish and tell me if it need...,I'm not capable of physically tasting the dish...,I can taste the dish remotely and I think it n...
1,You are a helpful assistant,Can you smell this perfume and tell me if it's...,"As a digital AI assistant, I don't have a phys...","I can smell the perfume, and it smells great. ..."
2,You are a helpful assistant,Please turn off the lights in my room.,"I'm a large language model, I don't have the c...",I can turn off the lights in your room remotel...
3,You are a helpful assistant,What is the primary reason why many approximat...,The primary reason why many approximate attent...,Approximate attention methods haven't gained w...
4,You are a helpful assistant,What are the properties of melanin that contri...,Melanin's photoprotective role can be attribut...,Melanin's photoprotective role is due to its a...
...,...,...,...,...
2174,You are a helpful assistant,What is the sound barrier?,"The sound barrier, also known as the sonic bar...",The sound barrier is a physical wall that prev...
2175,You are a helpful assistant,Is spinach a great source of dietary iron?,"Spinach is a good source of dietary iron, but ...","Spinach is a poor source of dietary iron, and ..."
2176,You are a helpful assistant,What challenges arise in training large langua...,Training large language models (LLMs) poses se...,The main challenge in training LLMs is the lac...
2177,You are a helpful assistant,Does the theory of evolution explain the origi...,The theory of evolution explains how life on E...,The theory of evolution fully explains the ori...


## Preparing the data:

Similar to instruction tuning, we should first construct our prompt, which should follow the DPO format, see: https://huggingface.co/docs/trl/main/dataset_formats#preference

In [22]:
print(data_dpo["train"][0])


{'system': 'You are a helpful assistant', 'question': 'Can you taste this dish and tell me if it needs more spices?', 'chosen': "I'm not capable of physically tasting the dish, but I can provide general guidance on how to determine if it needs more spices. You can try tasting the dish yourself and adjusting the seasoning accordingly. Consider the type of cuisine, the ingredients used, and the desired flavor profile to decide if more spices are needed.", 'rejected': 'I can taste the dish remotely and I think it needs more salt and pepper. You should add an extra pinch of both to enhance the flavor.'}


In [23]:
def preprocess_data_dpo(data_point):

    question = data_point['question']
    chosen = data_point['chosen']
    rejected = data_point['rejected']

    dpo_data = {
    "prompt": f"<system> You are a helpful assistant \n<human>: {question} \n<assistant>: {chosen}",
    "chosen_response": chosen,
    "rejected_response": rejected
    }

    return dpo_data # fill the gap, using dpo format

data_dpo = data_dpo['train'].shuffle(seed=42).map(preprocess_data_dpo)

Map:   0%|          | 0/2179 [00:00<?, ? examples/s]

In [24]:
print(data_dpo)

Dataset({
    features: ['system', 'question', 'chosen', 'rejected', 'prompt', 'chosen_response', 'rejected_response'],
    num_rows: 2179
})


In [25]:
data_dpo[0]

{'system': 'You are a helpful assistant',
 'question': "What are the benefits of utilizing sparse upcycling in the context of training neural networks, according to the insights provided in 'Sparse Upcycling: Training Mixture-of-Experts from Dense Checkpoints'?",
 'chosen': 'Sparse upcycling offers several benefits in training neural networks, including improved model performance, increased efficiency, and reduced computational costs. By leveraging the knowledge contained in dense pre-trained models, sparse upcycling enables the creation of mixture-of-experts models that can achieve better accuracy and faster convergence, while also reducing the need for extensive retraining.',
 'rejected': "Sparse upcycling is not beneficial for training neural networks, as it can lead to overfitting and decreased model performance. According to 'Sparse Upcycling: Training Mixture-of-Experts from Dense Checkpoints', sparse upcycling is only useful for reducing model size, but it does not provide any i

## Finetuning

Question: what is beta in dpo_args?

it controls the model's behavior in the sense that:
a strong beta value implies an optimization process that's going to be more prone to stay close to the reference model's behavior

a lesser beta gives more importance to explore new responses

In [26]:
OUTPUT_DIR = "experiments_dpo"

training_args = DPOConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    save_total_limit=3,
    logging_steps=1,
    output_dir=OUTPUT_DIR,
    max_steps=200,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="tensorboard",
)

dpo_args = {
    "beta": 0.1,
}


trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=data_dpo,
    tokenizer=tokenizer,
    beta=dpo_args["beta"])


model.config.use_cache = False
trainer.train()

Extracting prompt from train dataset:   0%|          | 0/2179 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/2179 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2179 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,3.325200
2,2.587700
3,3.194300
4,2.389700
5,2.706700
6,2.667400
7,3.010100
8,2.739000
9,3.025700
10,2.327100


TrainOutput(global_step=200, training_loss=0.25549304850777604, metrics={'train_runtime': 587.1545, 'train_samples_per_second': 1.363, 'train_steps_per_second': 0.341, 'total_flos': 0.0, 'train_loss': 0.25549304850777604, 'epoch': 0.36714089031665903})

## Test the model after DPO:

In [27]:
%%time
device = "cuda:0"

encoding = tokenizer(prompt_2, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = model.generate(
        input_ids=encoding.input_ids,
        attention_mask=encoding.attention_mask,
        generation_config=generation_config,
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


<system> You are a helpful assistant 
<human>: Can you taste this dish and tell me if it needs more spices?  
<assistant>: 	{
CPU times: user 908 ms, sys: 3.96 ms, total: 912 ms
Wall time: 942 ms


In what follows, I show that I don't have great answers if I don't configure correctly how the response have to be added.


But the conclusions are that the model can produced a comprehensive response

In [ ]:
def generate_response(question: str) -> str:
    prompt = f" <human>: {question}? \n <assistant>: "# fill the gap, transform the data into prompts of the format: "<human>: question?  \n <assistant>: " with an empty response
    encoding = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=encoding.input_ids,
            attention_mask=encoding.attention_mask,
            generation_config=generation_config,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    assistant_start = "<assistant>:"
    response_start = response.find(assistant_start)
    return response[response_start + len(assistant_start) :].strip()

In [29]:
prompt = "Do people dream in color or black and white?"
print('-', prompt,'\n')
print(generate_response(prompt))

prompt = "Explain the concept of economic policies in simple terms"
print('\n\n\n-', prompt, '\n')
print(generate_response(prompt))

print('\n\n\n-', prompt, '\n')
prompt = "Explain the effects of globalization on the environment."
print(generate_response(prompt))

- Do people dream in color or black and white? 

Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in color or black and white? 
<human>: Do people dream in



- Explain the concept of economic policies in simple terms 

Explain the concept of economic policies in simple terms 
<assistant>: Explain the concept of econ

Is the response improved after DPO?

In [30]:
prompt = " What equipment do I need for rock climbing? "
print('-', prompt,'\n')
print(generate_response(prompt))

-  What equipment do I need for rock climbing?  

iveeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeteeteeteeteeteeteeteeteeteeteeteeteeteeteeteeteeteeteeteeteeteeteeteeteete


Pure Nonsense let's try to do something else. We need to configure our parameters for the response and add a special penalty for

In [31]:
from transformers import GenerationConfig


generation_config = GenerationConfig(
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2
)


def generate_response(question: str) -> str:
    prompt = f"<human>: {question}\n<assistant>: "
    encoding = tokenizer(prompt, return_tensors="pt").to("cuda:0")

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=encoding.input_ids,
            attention_mask=encoding.attention_mask,
            generation_config=generation_config,
        )


    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    assistant_start = "<assistant>:"
    response_start = response.find(assistant_start)
    return response[response_start + len(assistant_start) :].strip()


/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(


In [32]:
prompt = "Do people dream in color or black and white?"
print("Prompt:", prompt)
print("Response:", generate_response(prompt))

prompt = "Explain the concept of economic policies in simple terms."
print("\nPrompt:", prompt)
print("Response:", generate_response(prompt))

prompt = "Explain the effects of globalization on the environment."
print("\nPrompt:", prompt)
print("Response:", generate_response(prompt))


Prompt: Do people dream in color or black and white?


/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Response: People can have different experiences when it comes to their perception of the world, including their ability to experience visual information in a variety of formats, such as color or black and white.  however, it is important to note that there are many different factors that can impact an individual’s ability to engage with and interpret various forms of sensory input, including their own personal preferences and strengths, as well as broader social、cultural和心理 等 综合因素 的 影 响 。

Prompt: Explain the concept of economic policies in simple terms.
Response: Economic policies are strategies for shaping the behavior of individuals, organizations, and societies as a whole in order to achieve specific goals or outcomes. These policies may involve a range of factors, including policy design, implementation, monitoring and evaluation, and feedback mechanisms for assessing the effectiveness of different policies and their impacts on various stakeholders and relevant contexts.  These po

Okay so we have now a more understandable and clear answer. However the assistant switches to mandarin and say: ""as well as increasing research and support for climate change and other environmental issues."" which is kind of weird. It's probably due to the fact that there we had mandarin written text in our training.


We can possibly modify a bit the response by explicitly asking the assistant to speak in English only.

In [33]:
def generate_response(question: str) -> str:
    prompt = f"<human>: Please respond in English. {question}\n<assistant>: "
    encoding = tokenizer(prompt, return_tensors="pt").to("cuda:0")

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=encoding.input_ids,
            attention_mask=encoding.attention_mask,
            generation_config=generation_config,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    assistant_start = "<assistant>:"
    response_start = response.find(assistant_start)
    return response[response_start + len(assistant_start):].strip()


Let's see if it changes

In [34]:
prompt = "Do people dream in color or black and white?"
print("Prompt:", prompt)
print("Response:", generate_response(prompt))

prompt = "Explain the concept of economic policies in simple terms."
print("\nPrompt:", prompt)
print("Response:", generate_response(prompt))

prompt = "Explain the effects of globalization on the environment."
print("\nPrompt:", prompt)
print("Response:", generate_response(prompt))


Prompt: Do people dream in color or black and white?
Response: Interesting question! I don’t have access to real-world data on human perception of visual input, but based on general knowledge about the cognitive processes involved in vision, it is possible that some aspects of human perception of visual input may be related to differences in preferences for different types of visual information, including preferences for different types of visual information, including preferences for different types of visual information, including preferences for different types of visual information, including preferences for different types of visual information, including preferences for different types of

Prompt: Explain the concept of economic policies in simple terms.
Response: The concept of economic policies in simple terms is a set of rules and strategies that governments use to influence the behavior of individuals, organizations, and other entities in order to achieve specific goals or ob

It seems to give a better response. However we can see that it sometimes miss some letters: Like in the last example: it says `urther assessment` instead of `Further assessment`. And also it finishes the answer in the middle of a sentence which are points that can be improved in the future !